# lora-stack-lab Kaggle runner
Set `BRANCH`/`REPO`, enable **Internet** in notebook settings, run **Save & Run All (Commit)**.
Outputs: `results/` JSONs + `ckpt/` (small) + `results_pack.zip` for manual download.

In [ ]:
import os, subprocess, sys

BRANCH  = "main"                # <- change per stage
STAGE   = "stage2"               # <- "reeval" (score saved adapters) or stage prefix of configs/experiments/
REPO    = "P-Bao/lora-stack-lab"  # public repo
COMMIT  = None                    # <- optional: pin exact commit for reproduction
RUN_IDS = ["E0-SMOKE-G-qwen05b-lr2e-4-r32-s1", "E0-SMOKE-Bp-qwen05b-lr2e-4-r32-s1",
           "stage2-G-lr2e-4-r32-s1", "stage2-G-lr1e-4-r32-s1", "stage2-G-lr5e-4-r32-s1",
           "stage2-Bp-lr2e-4-r32-s1", "stage2-Bp-lr1e-4-r32-s1", "stage2-Bp-lr5e-4-r32-s1"]  # session A: smokes + G/Bp (docs/stage2-plan.md)
# Session B (docs/stage2-plan.md): swap RUN_IDS for these - smokes first (suffix -qwen05b-lr2e-4-r32-s1):
#   "E0-SMOKE-O", "E0-SMOKE-Oi", "E0-SMOKE-Br", "E0-SMOKE-A",
#   stage2-O-lr1e-4, O-lr5e-5, O-lr2e-4, Oi-lr1e-4, Oi-lr2e-4,
#   Br-lr1e-4, Br-lr2e-4, A-lr2e-4, A-lr1e-3, A-lr5e-3, A-lr1e-2   (all -r32-s1)

sh = lambda c: subprocess.run(c, shell=True, check=True)
sh("rm -rf /tmp/repo")  # idempotent: always clone fresh (stale /tmp/repo = stale code)
sh(f"git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git /tmp/repo")
if COMMIT:
    sh(f"cd /tmp/repo && git fetch --depth 1 origin {COMMIT} && git checkout {COMMIT}")
os.chdir("/tmp/repo")
sh("bash scripts/bootstrap_kaggle.sh")
sys.path.insert(0, "/tmp/repo/src")

In [ ]:
# STAGE="reeval": re-evaluate saved adapters (no training). Dataset dirs: <run_id>/adapter/
# Kaggle mount styles: /kaggle/input/<slug> (old) or /kaggle/input/datasets/<user>/<slug> (new).
from pathlib import Path
import glob as _glob

ADAPTER_DATASET = "stage1-adapters"   # <- dataset slug; contains <run_id>/adapter/
ADAPTER_ROOTS = [f"/kaggle/input/{ADAPTER_DATASET}",
                 *_glob.glob(f"/kaggle/input/datasets/*/{ADAPTER_DATASET}")]
ADAPTER_ROOT = next((p for p in ADAPTER_ROOTS if Path(p).exists()), None)
print("[nb] adapter root:", ADAPTER_ROOT, "| candidates:", ADAPTER_ROOTS, flush=True)

def run_reeval(adapter_root, eval_size=None):
    """One reeval shard per T4 (reeval.py --shard i/n takes runs with index % n == i).
    A failed shard only costs a rerun: finished runs are skipped. Returns exit codes."""
    procs = []
    for gpu in (0, 1):
        cmd = [sys.executable, "-u", "scripts/reeval.py",
               "--adapters", adapter_root, "--out", "/kaggle/working/results",
               "--batch-size", "16", "--shard", f"{gpu}/2"]
        if eval_size:
            cmd += ["--eval-size", str(eval_size)]
        procs.append(subprocess.Popen(cmd, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}))
        print(f"[nb] reeval shard {gpu}/2 -> GPU {gpu}", flush=True)
    codes = [pr.wait() for pr in procs]
    if any(codes):
        print(f"[nb] WARNING: reeval shards failed with codes {codes} - fix and rerun; "
              f"finished runs are skipped on rerun", flush=True)
    return codes


if STAGE == "reeval":
    assert ADAPTER_ROOT, f"dataset '{ADAPTER_DATASET}' not mounted - add it under Add Input"
    rc = 1 if any(run_reeval(ADAPTER_ROOT, eval_size=200)) else 0
else:
    ids = " ".join(RUN_IDS)
    # run_queue exits non-zero when some runs fail (e.g. OOM) - never abort the cell here:
    # reeval + collect must still run for the runs that did finish (a failed session
    # otherwise loses both, which is exactly what the Bp OOM session did).
    rc = subprocess.run(f"python scripts/run_queue.py --stage {STAGE} --ids {ids} "
                        f"--out /kaggle/working/results --ckpt /kaggle/working/ckpt "
                        f"--skip-from /kaggle/input", shell=True).returncode  # skips done IDs
    if rc:
        print(f"[nb] WARNING: run_queue exit={rc} (see <id>.FAILED.json); "
              f"reeval + collect continue for finished runs", flush=True)
    # reeval the fresh step-500 snapshots so cross-seed compare is fair vs s1 (trap 10)
    run_reeval("/kaggle/working/ckpt")
sh("python scripts/collect_results.py --src /kaggle/working/results "
   "--zip /kaggle/working/results_pack.zip")
assert rc == 0, f"run_queue/reeval failed (exit {rc}) - see the log above"


In [ ]:
sh("python scripts/collect_results.py --src /kaggle/working/results --zip /kaggle/working/results_pack.zip")